# КЕДР: вид DRP против озера за июль–октябрь 2026

Старой таблицы за эти месяцы нет, поэтому эталон считаем заново из озера: `Kedr.v_detail_dmkb`, `Kedr.v_detail_dmsb`, `Kedr.v_detail_dkb` (как строилась старая таблица: сумма `nbi_ssp` по клиенту в каждом виде, потом максимум из трёх).

1. Какие месяцы есть в озере и в виде.
2. Вид DRP: клиенты, строки, продукты, сумма по месяцам.
3. Озеро: ЧОД по тем же клиентам (запросы пачками по ИНН, чтобы не сканировать месяц целиком).
4. Сверка вида с озером по клиенту и месяцу.
5. Все ли клиенты витрины (история `sandbox_ai.shestopalov_acq_final_df_hist`) есть в виде.

Только чтение. Логины спрашиваются один раз. Если кластер занят, запросы к озеру сами повторятся через 3 минуты.

In [ ]:
import re
import time
from getpass import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

DRP_VIEW = "sbx_edm_dfip.v_detail_dmkb_dkb_dmsb"
LAKE_VIEWS = {"dmkb": "Kedr.v_detail_dmkb", "dmsb": "Kedr.v_detail_dmsb", "dkb": "Kedr.v_detail_dkb"}
HIST_TABLE = "sandbox_ai.shestopalov_acq_final_df_hist"
MONTHS = [202607, 202608, 202609, 202610]
CHUNK = 600

if "drp" not in globals():
    drp = connect(
        to="DRP",
        user_params={"user_name": input("DRP login: ").strip(), "password": getpass("DRP password: ")},
    )

if "imp" not in globals():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={"user_name": input("Login (keytab): ").strip()},
    )


def drp_fetch(sql):
    with drp:
        return drp.fetch(sql)


STALE_MARKERS = ("Failed to open HDFS file", "File does not exist")


def refresh_lake():
    for view in LAKE_VIEWS.values():
        try:
            with imp:
                imp.execute(f"REFRESH {view}")
            print(f"  REFRESH {view}: ok")
        except Exception as exc:
            print(f"  REFRESH {view}: не удалось ({str(exc)[:200]})")


def imp_fetch(sql, attempts=4, wait_sec=180):
    for attempt in range(1, attempts + 1):
        try:
            with imp:
                imp.execute("SET MEM_LIMIT=4g")
                return imp.fetch(sql)
        except Exception as exc:
            if any(m in str(exc) for m in STALE_MARKERS) and attempt < attempts:
                print("  устаревшие метаданные в Impala, делаю REFRESH")
                refresh_lake()
                continue
            if "Admission for query exceeded timeout" in str(exc) and attempt < attempts:
                print(f"  кластер занят, жду {wait_sec} с (попытка {attempt})")
                time.sleep(wait_sec)
                continue
            raise


def norm_inn(v):
    s = "".join(ch for ch in str(v) if ch.isdigit())
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s or None


def months_sql(col_type):
    if "int" in col_type:
        return ", ".join(str(m) for m in MONTHS)
    return ", ".join(f"'{m}'" for m in MONTHS)

## 1. Какие месяцы есть в озере и в виде

In [ ]:
lake_types = {}
for src, view in LAKE_VIEWS.items():
    desc = imp_fetch(f"DESCRIBE {view}")
    types = dict(zip(desc.iloc[:, 0].str.lower(), desc.iloc[:, 1].str.lower()))
    lake_types[src] = {"yearmm": types.get("yearmm", ""), "inn": types.get("inn", "")}
    mx = imp_fetch(f"SELECT max(yearmm) AS max_yearmm FROM {view}")
    print(f"{view}: max yearmm = {mx.max_yearmm.iloc[0]} | тип yearmm = {lake_types[src]['yearmm']} | тип inn = {lake_types[src]['inn']}")

view_months = drp_fetch(f"""
    SELECT yearmm,
           count(*) AS rows_cnt,
           count(DISTINCT inn) AS inn_cnt,
           count(DISTINCT CASE WHEN prod_name = 'Торговый эквайринг' THEN inn END) AS inn_with_acq,
           count(DISTINCT prod_name) AS prod_names,
           sum(nbi_ssp) AS nbi
    FROM {DRP_VIEW}
    WHERE yearmm >= 202601
    GROUP BY yearmm
    ORDER BY yearmm
""")
print("\nВид DRP по месяцам 2026:")
display(view_months)

## 2. Вид DRP: ЧОД по клиентам за проверяемые месяцы

In [ ]:
view_inn = drp_fetch(f"""
    SELECT yearmm, inn, sum(nbi_ssp) AS view_value
    FROM {DRP_VIEW}
    WHERE yearmm IN ({', '.join(str(m) for m in MONTHS)})
    GROUP BY yearmm, inn
""")
view_inn["inn"] = view_inn["inn"].map(norm_inn)
view_inn["yearmm"] = view_inn["yearmm"].astype(int)
view_inn["view_value"] = pd.to_numeric(view_inn["view_value"], errors="coerce").fillna(0)
view_inn = view_inn.groupby(["yearmm", "inn"], as_index=False)["view_value"].sum()

report_months = ", ".join("'" + str(m)[:4] + "-" + str(m)[4:] + "'" for m in MONTHS)
hist = imp_fetch(f"""
    SELECT DISTINCT report_month, cast(inn AS string) AS inn
    FROM {HIST_TABLE}
    WHERE report_month IN ({report_months})
      AND inn IS NOT NULL
""")
hist["yearmm"] = hist["report_month"].str.replace("-", "").astype(int)
hist["inn"] = hist["inn"].map(norm_inn)
hist = hist.dropna(subset=["inn"]).drop_duplicates(["yearmm", "inn"])

print("Клиентов в виде по месяцам:", view_inn.groupby("yearmm").size().to_dict())
print("Клиентов витрины в истории по месяцам:", hist.groupby("yearmm").size().to_dict())

## 3. Озеро: ЧОД по тем же клиентам

Берём ИНН из вида и из витрины, идём пачками по 600 ИНН. В список кладём ИНН и с ведущими нулями, и без них. Может занять 10–20 минут.

In [ ]:
all_inn = sorted(set(view_inn["inn"]) | set(hist["inn"]))
print("ИНН для запроса к озеру:", len(all_inn))

lake_parts = []
for src, view in LAKE_VIEWS.items():
    ym_list = months_sql(lake_types[src]["yearmm"])
    for i in range(0, len(all_inn), CHUNK):
        chunk = all_inn[i:i + CHUNK]
        variants = sorted(set(chunk) | {x.lstrip("0") for x in chunk if x.lstrip("0")})
        inn_list = ", ".join(f"'{x}'" for x in variants)
        df = imp_fetch(f"""
            SELECT cast(yearmm AS bigint) AS yearmm, cast(inn AS string) AS inn, sum(nbi_ssp) AS nbi
            FROM {view}
            WHERE yearmm IN ({ym_list})
              AND cast(inn AS string) IN ({inn_list})
            GROUP BY cast(yearmm AS bigint), cast(inn AS string)
        """)
        if df is not None and len(df):
            df["src"] = src
            lake_parts.append(df)
    print(f"{view}: готово")

lake = pd.concat(lake_parts, ignore_index=True) if lake_parts else pd.DataFrame(columns=["yearmm", "inn", "nbi", "src"])
lake["inn"] = lake["inn"].map(norm_inn)
lake["yearmm"] = lake["yearmm"].astype(int)
lake["nbi"] = pd.to_numeric(lake["nbi"], errors="coerce").fillna(0)
lake_by_src = lake.groupby(["yearmm", "inn", "src"], as_index=False)["nbi"].sum()
lake_inn = (
    lake_by_src.groupby(["yearmm", "inn"], as_index=False)
    .agg(lake_max=("nbi", "max"), lake_sum=("nbi", "sum"), src_cnt=("src", "nunique"))
)
print("Озеро по месяцам (клиентов):", lake_inn.groupby("yearmm").size().to_dict())

## 4. Сверка вида с озером

Хорошо, если `match_pct` около 99–100% и суммы `view_total` и `lake_total` почти равны.

In [ ]:
cmp = view_inn.merge(lake_inn, on=["yearmm", "inn"], how="outer", indicator=True)
both = cmp[cmp["_merge"] == "both"].copy()
both["match"] = (both["view_value"] - both["lake_max"]).abs() < 1

summary = (
    cmp.groupby("yearmm")
    .apply(lambda g: pd.Series({
        "inn_view": int((g["_merge"] != "right_only").sum()),
        "inn_lake": int((g["_merge"] != "left_only").sum()),
        "inn_both": int((g["_merge"] == "both").sum()),
        "only_view": int((g["_merge"] == "left_only").sum()),
        "only_lake": int((g["_merge"] == "right_only").sum()),
    }))
    .join(both.groupby("yearmm").agg(
        match_pct=("match", lambda s: round(100 * s.mean(), 2)),
        view_total=("view_value", "sum"),
        lake_total=("lake_max", "sum"),
        multi_src=("src_cnt", lambda s: int((s > 1).sum())),
    ))
    .reset_index()
)
display(summary)

both["diff"] = both["view_value"] - both["lake_max"]
print("Топ-20 расхождений:")
display(
    both.loc[~both["match"]]
    .assign(abs_diff=lambda d: d["diff"].abs())
    .sort_values("abs_diff", ascending=False)
    [["yearmm", "inn", "src_cnt", "view_value", "lake_max", "lake_sum", "diff"]]
    .head(20)
)

## 5. Все ли клиенты витрины есть в виде

`in_view_pct` — доля клиентов витрины, у которых в виде есть ЧОД за месяц. `in_lake_not_view` — есть в озере, но нет в виде (их вид теряет).

In [ ]:
cov = hist[["yearmm", "inn"]].copy()
cov = cov.merge(view_inn[["yearmm", "inn"]].assign(in_view=1), on=["yearmm", "inn"], how="left")
cov = cov.merge(lake_inn[["yearmm", "inn"]].assign(in_lake=1), on=["yearmm", "inn"], how="left")
cov[["in_view", "in_lake"]] = cov[["in_view", "in_lake"]].fillna(0).astype(int)

coverage = cov.groupby("yearmm").agg(
    mart_inn=("inn", "size"),
    in_view=("in_view", "sum"),
    in_lake=("in_lake", "sum"),
    in_lake_not_view=("in_lake", lambda s: int(((s == 1) & (cov.loc[s.index, "in_view"] == 0)).sum())),
).reset_index()
coverage["in_view_pct"] = (100 * coverage["in_view"] / coverage["mart_inn"]).round(2)
coverage["in_lake_pct"] = (100 * coverage["in_lake"] / coverage["mart_inn"]).round(2)
pd.set_option("display.float_format", "{:,.2f}".format)
display(coverage)

lost = cov[(cov["in_lake"] == 1) & (cov["in_view"] == 0)].merge(lake_inn, on=["yearmm", "inn"])
print("Топ-20 клиентов витрины, которые есть в озере, но нет в виде (по модулю ЧОД):")
display(lost.assign(abs_v=lost["lake_max"].abs()).sort_values("abs_v", ascending=False)
        [["yearmm", "inn", "src_cnt", "lake_max"]].head(20))